# DarkPipe 0.5 — validación AION con autoridad observacional

Este Colab reproduce la campaña congelada `DP-AION-0.4-20260825` y añade el recibo tipado de DarkPipe 0.5. Los endpoints físicos permanecen iguales; la nueva capa impide promover observaciones a causalidad, detección, generalización o intervención.


In [ ]:
from pathlib import Path
import os, subprocess, sys

in_colab = Path("/content").exists() and "google.colab" in sys.modules
if in_colab:
    repo = Path("/content/darkpipe-realdata")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/FacundoFirmenich/darkpipe-realdata.git", str(repo)], check=True)
    else:
        print("Se usa el checkout ya presente:", repo)
else:
    repo = Path.cwd()
    print("Ejecución local desde:", repo)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[test]"], check=True)

## 1. Verificación de versión y jurisdicción de licencias

In [ ]:
import darkpipe
print("DarkPipe", darkpipe.__version__)
assert darkpipe.__version__ == "0.5.0"
project_notice = Path("LICENSE-NOTICE").read_text(encoding="utf-8")
upstream_notice = Path("evidence/aion_sensor_validation_2026-08-25/UPSTREAM_NOTICE.md").read_text(encoding="utf-8")
assert "GPL-3.0-or-later" in project_notice and "any later version" in project_notice
assert "CC-BY-4.0" in upstream_notice and "MIT" in upstream_notice
print("Licencias verificadas: DarkPipe GPL-3.0-or-later; evidencia AION CC-BY-4.0/MIT según ámbito.")

## 2. Suite completa

Rehashea la evidencia, reproduce los endpoints y prueba las barreras de autoridad.


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)


## 3. Ejecución del protocolo congelado

In [ ]:
from darkpipe.aion import run_aion_validation

evidence = Path("evidence/aion_sensor_validation_2026-08-25")
output = Path("/content/darkpipe_aion_v05_run") if in_colab else Path("work/colab_aion_v05_run")
report = run_aion_validation(evidence, output)
print({
    "decision": report["decision"],
    "gate_0": report["gate_0"]["passed"],
    "E1": f"{report['endpoint_e1']['passed_count']}/7",
    "E2": report["endpoint_e2"]["passed"],
})
assert report["decision"] == "PASS_BOUNDED"

## 4. Resultado sustantivo y tabla de recuperaciones

In [ ]:
import pandas as pd
recovery = pd.DataFrame(report["endpoint_e1"]["datasets"])[[
    "dataset_id", "truth_frequency_hz", "recovered_frequency_hz",
    "fourier_resolution_hz", "resolution_normalized_error", "passed"
]]
recovery

In [ ]:
e2 = report["endpoint_e2"]
pd.Series({
    "HLN_minus_LLN_microrad": 1e6 * e2["difference_rad"],
    "combined_uncertainty_microrad": 1e6 * e2["difference_uncertainty_rad"],
    "CI95_low_microrad": 1e6 * e2["ci95_rad"][0],
    "CI95_high_microrad": 1e6 * e2["ci95_rad"][1],
    "z_score": e2["z_score"],
    "passes_frozen_rule": e2["passed"],
})

## 5. Autoridad observacional ejecutable

La tabla conserva los claims amplios como `NOT_ESTIMABLE`; no los elimina ni los promociona.


In [ ]:
authority = report["authority"]
claims = pd.DataFrame(authority["claim_ledger"]["records"])[[
    "claim_id", "kind", "status", "future_only"
]]
assert authority["claim_ledger"]["automatic_promotion"] is False
assert claims.loc[claims.claim_id == "aion.dark_sector_detection", "status"].item() == "NOT_ESTIMABLE"
claims


## 6. Informe y figura comprobados

In [ ]:
try:
    from IPython.display import Image, Markdown, display
    display(Markdown((output / "report.md").read_text(encoding="utf-8")))
    display(Image(filename=str(output / "validation.png")))
except ImportError:
    assert (output / "report.md").is_file() and (output / "validation.png").is_file()
    print("Recibo verificado:", output / "report.md")
    print("Figura verificada:", output / "validation.png")


## 7. Recibo liviano para descargar

El ZIP contiene sólo reporte, figura y manifiesto; no duplica los 19 MB de evidencia AION.

In [ ]:
import shutil
archive_base = Path("/content/DarkPipe_Authority_0.5_receipt") if in_colab else Path("work/DarkPipe_Authority_0.5_receipt")
zip_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=output))
print(zip_path, zip_path.stat().st_size, "bytes")
if in_colab:
    from google.colab import files
    files.download(str(zip_path))

## Interpretación final

`PASS_BOUNDED` significa que el conjunto AION seleccionado superó integridad, recuperación 7/7 y consistencia HLN–LLN según reglas congeladas. No autoriza una afirmación de detección, exclusión ni transferencia a AION-10/AION-km.